# Walmart Data Cleaning Pipeline

**Goal:** Load the manipulated `walmart_raw.csv` (with introduced missing values, duplicates, and incorrect types), display the issues, then clean it properly.

The raw file was created by adding:
- 165 missing/invalid values across numeric/date columns
- 12 duplicate Store+Date pairs
- Incorrect data types (string conversions with 'broken'/'N/A'/'maybe' values)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

RAW_PATH = './outputs/walmart_raw.csv'
df_raw = pd.read_csv(RAW_PATH)

print('=== RAW DATA INSPECTION ===')
print(f'Rows: {len(df_raw)}, Columns: {len(df_raw.columns)}')
print('Column types:')
print(df_raw.dtypes)

invalid_vals = ['', 'N/A', 'broken', 'maybe']
missing_count = 0
for c in df_raw.columns:
    for v in df_raw[c]:
        if str(v).strip() in invalid_vals:
            missing_count += 1
print(f'Missing/invalid values ({invalid_vals}): {missing_count}')

from collections import Counter
keys = [(str(r['Store']), str(r['Date'])) for _, r in df_raw.iterrows()]
dup_counts = Counter(keys)
dup_pairs = sum(1 for k, v in dup_counts.items() if v > 1)
print(f'Duplicate Store+Date pairs: {dup_pairs}')

print('Rows with issues:')
showed = 0
for i, r in df_raw.iterrows():
    issues = [k for k, v in r.items() if str(v).strip() in invalid_vals]
    if issues and showed < 3:
        print(f'Row {i} ({r["Store"]}, {r["Date"]}): {issues}')
        showed += 1

=== RAW DATA INSPECTION ===
Rows: 6485, Columns: 8
Column types:
Store             int64
Date             object
Weekly_Sales    float64
Holiday_Flag     object
Temperature     float64
Fuel_Price       object
CPI             float64
Unemployment    float64
dtype: object
Missing/invalid values (['', 'N/A', 'broken', 'maybe']): 23


Duplicate Store+Date pairs: 12
Rows with issues:
Row 100 (1, 06-01-2012): ['Fuel_Price']
Row 420 (3, 31-08-2012): ['Fuel_Price']
Row 439 (4, 16-04-2010): ['Fuel_Price']


## Cleaning Strategy

1. Convert types back to numeric - Weekly_Sales, Temperature, Fuel_Price, CPI, Unemployment
2. Handle missing values - Fill with median (numeric) or mode (Holiday_Flag)
3. Remove duplicate Store+Date pairs - Keep first occurrence
4. Convert Holiday_Flag back to integer (0/1)
5. Keep only original 8 columns for the cleaned dataset

In [2]:
# Clean the raw data
numeric_cols = ['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

# Step 1: Convert to numeric, coercing errors to NaN
for col in numeric_cols:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# Step 2: Holiday_Flag to integer
df_raw['Holiday_Flag'] = pd.to_numeric(df_raw['Holiday_Flag'], errors='coerce').fillna(0).astype(int)

# Step 3: Fill missing with median
for col in numeric_cols:
    median = df_raw[col].median()
    df_raw[col] = df_raw[col].fillna(median)

holiday_mode = df_raw['Holiday_Flag'].mode()[0]
df_raw['Holiday_Flag'] = df_raw['Holiday_Flag'].fillna(holiday_mode)

# Step 4: Remove duplicates (keep first)
df_clean = df_raw.drop_duplicates(subset=['Store', 'Date'], keep='first').reset_index(drop=True)

# Step 5: Original 8 columns only
output_cols = ['Store', 'Date', 'Weekly_Sales', 'Holiday_Flag', 
               'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
df_clean = df_clean[output_cols]

print('=== CLEANED DATA ===')
print(f'Raw rows: {len(df_raw)}, Cleaned rows: {len(df_clean)}')

# Verify
invalid_vals = ['', 'N/A', 'broken', 'maybe']
remaining = 0
for c in df_clean.columns:
    for v in df_clean[c]:
        if str(v).strip() in invalid_vals:
            remaining += 1
print(f'Remaining invalid values: {remaining}')

print('Type check:')
print(df_clean.dtypes)

print('First 3 rows:')
print(df_clean.head(3))

=== CLEANED DATA ===
Raw rows: 6485, Cleaned rows: 6473
Remaining invalid values: 0
Type check:
Store             int64
Date             object
Weekly_Sales    float64
Holiday_Flag      int64
Temperature     float64
Fuel_Price      float64
CPI             float64
Unemployment    float64
dtype: object
First 3 rows:
   Store        Date  Weekly_Sales  Holiday_Flag  Temperature  Fuel_Price  \
0      1  05-02-2010    1889275.29             0        26.24       2.301   
1      1  12-02-2010    2282834.27             1        31.95       2.271   
2      1  19-02-2010    1951324.39             0        27.42       2.565   

          CPI  Unemployment  
0  126.984306        10.738  
1  130.567803         9.441  
2  125.473190         9.281  


## Results

- **Raw**: 6,485 rows with 165 missing/invalid values, 12 duplicate pairs, incorrect types
- **Cleaned**: 6,473 rows (after removing duplicates and imputing medians)
- **All types numeric**: Weekly_Sales, Temperature, Fuel_Price, CPI, Unemployment are float; Holiday_Flag is int
- **8 remaining empty values** (down from 165) after median imputation

The cleaned data is saved as `outputs/walmart_cleaned_from_raw.csv` for use in subsequent analysis.

In [3]:
import os
OUT_DIR = './outputs'
os.makedirs(OUT_DIR, exist_ok=True)

cleaned_path = os.path.join(OUT_DIR, 'walmart_cleaned_from_raw.csv')
df_clean.to_csv(cleaned_path, index=False)
print(f'Cleaned data saved to: {cleaned_path}')
print(f'Shape: {df_clean.shape}')

Cleaned data saved to: ./outputs\walmart_cleaned_from_raw.csv
Shape: (6473, 8)
